# DOAgent — Minimal demo

This notebook runs a **minimal** DOAgent example step by step. It only uses the `doagent` library and code defined in this notebook. Run it in Google Colab or any environment with Python 3.10+.

**What you'll do:** Install doagent → create a session → define a tiny environment and policy → run one step → inspect recorded data.

## Step 1 — Install the library

Install DOAgent from the repository. This is the only dependency you need for this minimal demo.

In [ ]:
!pip install -q git+https://github.com/cabrerac/doagent.git

## Step 2 — Import doagent

Import the Session API. All recording is handled by the library.

In [ ]:
from doagent import Session

## Step 3 — Create a session from config

Build a config dict and create a session. The session is configured with **in-memory as the shared data model**: records live in memory so we can inspect them after the run. We register a single policy (a no-op that always returns action 0).

In [ ]:
config = {
    "shared_data": {"type": "memory"},
    "run_config": {"logging_level": 2},
    "topology": {"mode": "centralised"},
    "policies": {
        "noop": lambda params: lambda req: {"choice": {"status": "act", "action": 0}},
    },
}
session = Session.from_config(config)

## Step 4 — Define a minimal environment

Your environment must provide:
- `reset(seed=...)` → observations dict (one key per agent)
- `step(actions)` → dict with `observations`, `rewards`, and `terminations` (or `done`)
- `agents` (property or attribute) → list of agent ids

Here we use a stub env that returns a single agent and trivial observations.

In [ ]:
class StubEnv:
    agents = ["agent-1"]

    def reset(self, *, seed=None):
        return {"agent-1": {"text": "Hello from DOAgent"}}

    def step(self, actions):
        return {
            "observations": {"agent-1": {"text": "Step complete"}},
            "rewards": {"agent-1": 0.0},
            "terminations": {"agent-1": False},
        }

## Step 5 — Wrap the env and create agents

The library wraps your env so that each `step` is recorded (outcomes, traces). Then we create one agent that uses the "noop" policy.

In [ ]:
env = session.wrap_env(StubEnv(), env_actor="stub_env")
agents = session.create_agents(
    [{"id": "agent-1", "policy": {"name": "noop", "params": {}}}],
    goal="demo",
    payload_type="demo_update",
)

## Step 6 — Run one step

Reset the env, ask the agent for a decision, then step. Recording happens inside the library.

In [ ]:
observations = env.reset(seed=42)
result = agents["agent-1"].decide(observations["agent-1"], 1, inputs={})
env.step({"agent-1": result["action"]})
print("Step completed.")

## Step 7 — Inspect recorded data

Use `session.inspect(kind)` to read what was recorded: `agent_update` (decisions) and `outcome` (state after step).

In [ ]:
agent_updates = session.inspect("agent_update")
outcomes = session.inspect("outcome")
print(f"Recorded {len(agent_updates)} agent_update(s), {len(outcomes)} outcome(s).")
for r in agent_updates:
    choice = r.payload.get("decision", {}).get("response", {}).get("choice", {})
    print(f"  - {r.id}: status={choice.get('status')}, action={choice.get('action')}")

---
**Next:** Try the [Push demo](02_push_demo.ipynb) (PettingZoo) or the [Grid-world demo](03_gridworld_demo.ipynb) (session with file as shared data model + analysis).

For interpreting generated analysis artefacts and plots, see [guides/interpreting-analysis.md](https://github.com/cabrerac/doagent/blob/main/guides/interpreting-analysis.md) (local path: `guides/interpreting-analysis.md`). The current analysis tools are an expandable demonstration set of what DOAgent analysis enables.